# Langkah 1: Persiapan Rilis


In [1]:
from pathlib import Path
from datetime import datetime
import json

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Rectangle
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 140)

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
elif not (project_root / "data").exists():
    project_root = project_root.parent

models_dir = project_root / "models"
reports_dir = project_root / "reports"
diagrams_dir = project_root / "diagrams"
api_dir = project_root / "api"
notebooks_dir = project_root / "notebooks"

for folder in [models_dir, reports_dir, diagrams_dir, api_dir, notebooks_dir]:
    folder.mkdir(parents=True, exist_ok=True)

model_path = models_dir / "baseline_price_model.joblib"
quality_checklist_path = reports_dir / "final_quality_checklist.md"
model_evaluation_report_path = reports_dir / "final_quality_checklist.md"  # Gabung ke satu file
registry_path = models_dir / "model_registry.json"
cd_report_path = reports_dir / "cd_scenario_report.md"
pipeline_design_report_path = reports_dir / "mlops_pipeline_design.md"
diagram_md_path = diagrams_dir / "mlops_pipeline_diagram.md"
diagram_png_path = diagrams_dir / "mlops_pipeline_diagram.png"

print("Project root:", project_root)







Project root: D:\SEMESTER 6\DATA MINING\MLOps-Tarif-Transportasi-Online


In [2]:
model_file_available = model_path.exists()
ci_checklist_available = quality_checklist_path.exists()
ct_checklist_available = quality_checklist_path.exists()
model_performance_available = model_evaluation_report_path.exists()
ready_for_cd_design = all([
    model_file_available,
    ci_checklist_available,
    ct_checklist_available,
    model_performance_available,
])

readiness_summary = pd.DataFrame({
    "Readiness Item": [
        "Model file available",
        "CI checklist available",
        "CT checklist available",
        "Model performance available",
        "Ready for CD scenario design",
    ],
    "Status": [
        "YES" if model_file_available else "NO",
        "YES" if ci_checklist_available else "NO",
        "YES" if ct_checklist_available else "NO",
        "YES" if model_performance_available else "NO",
        "YES" if ready_for_cd_design else "NO",
    ],
    "Path / Note": [
        str(model_path),
        str(quality_checklist_path),
        str(quality_checklist_path),
        str(model_evaluation_report_path),
        "Proceed with local CD design only" if ready_for_cd_design else "Complete earlier steps first",
    ],
})

display(readiness_summary)







,Readiness Item,Status,Path / Note
0,Model file available,YES,D:\SEMESTER 6\DATA MINING\MLOps-Tarif-Transpor...
1,CI checklist available,YES,D:\SEMESTER 6\DATA MINING\MLOps-Tarif-Transpor...
2,CT checklist available,YES,D:\SEMESTER 6\DATA MINING\MLOps-Tarif-Transpor...
3,Model performance available,YES,D:\SEMESTER 6\DATA MINING\MLOps-Tarif-Transpor...
4,Ready for CD scenario design,YES,Proceed with local CD design only


# Langkah 2: Model Registry


In [3]:
model_metadata = {
    "model_name": "estimator_tarif_transportasi",
    "model_version": "v1.0-baseline-random-forest",
    "model_type": "Random Forest Regressor",
    "model_file": "models/baseline_price_model.joblib",
    "target_column": "price",
    "problem_type": "regression",
    "features_used": [
        "distance",
        "cab_type",
        "source",
        "destination",
        "name",
        "hour",
        "day",
        "month",
        "day_of_week",
    ],
    "features_excluded": {
        "id": "identifier",
        "product_id": "identifier atau kode produk",
        "price": "kolom target",
        "surge_multiplier": "terlalu terkait dengan mekanisme penentuan harga",
        "weather_columns": "belum digabung karena penyesuaian waktu dan lokasi belum disiapkan",
    },
    "training_dataset": "data/processed/cleaned_cab_rides.csv",
    "rows_after_cleaning": 637976,
    "MAE": 1.4254,
    "RMSE": 2.6181,
    "R2_score": 0.9214,
    "CI_status": "PASS",
    "CT_status": "PASS, warning pada error grup service name",
    "approval_status": "Disetujui untuk simulasi shadow deployment",
    "warning_notes": [
        "Grup layanan dengan error tertinggi: Lux Black XL.",
        "Gunakan shadow deployment dan pantau MAE per grup sebelum promosi.",
        "Model ini simulasi pembelajaran, bukan sistem pricing produksi.",
    ],
    "created_at": datetime.now().isoformat(timespec="seconds"),
}

registry = {"registered_models": [model_metadata]}
registry_path.write_text(json.dumps(registry, indent=2), encoding="utf-8")
print("Registry model disimpan di:", registry_path)
display(pd.DataFrame([model_metadata]))







Registry model disimpan di: D:\SEMESTER 6\DATA MINING\MLOps-Tarif-Transportasi-Online\models\model_registry.json


,model_name,model_version,model_type,model_file,target_column,problem_type,features_used,features_excluded,training_dataset,rows_after_cleaning,MAE,RMSE,R2_score,CI_status,CT_status,approval_status,warning_notes,created_at
0,estimator_tarif_transportasi,v1.0-baseline-random-forest,Random Forest Regressor,models/baseline_price_model.joblib,price,regression,"[distance, cab_type, source, destination, name...","{'id': 'identifier', 'product_id': 'identifier...",data/processed/cleaned_cab_rides.csv,637976,1.4254,2.6181,0.9214,PASS,"PASS, warning pada error grup service name",Disetujui untuk simulasi shadow deployment,[Grup layanan dengan error tertinggi: Lux Blac...,2026-06-15T07:40:58


# Langkah 3: Setup API Lokal


In [4]:
app_code = """from pathlib import Path
import sys

import joblib
import pandas as pd
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel


PROJECT_ROOT = Path(__file__).resolve().parents[1]
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))

from preprocessing import create_time_features  # noqa: E402


MODEL_VERSION = "v1.0-baseline-random-forest"
MODEL_PATH = PROJECT_ROOT / "models" / "baseline_price_model.joblib"
FEATURE_COLUMNS = [
    "distance",
    "cab_type",
    "source",
    "destination",
    "name",
    "hour",
    "day",
    "month",
    "day_of_week",
]


app = FastAPI(
    title="Online Transportation Fare Estimation API",
    description="Local learning simulation API for an MLOps mini project.",
    version=MODEL_VERSION,
)

model = joblib.load(MODEL_PATH)


class FarePredictionRequest(BaseModel):
    distance: float
    cab_type: str
    source: str
    destination: str
    name: str
    time_stamp: int


def validate_request(data):
    if data.distance <= 0:
        raise HTTPException(status_code=400, detail="distance must be greater than 0.")

    text_fields = {
        "cab_type": data.cab_type,
        "source": data.source,
        "destination": data.destination,
        "name": data.name,
    }
    for field_name, value in text_fields.items():
        if not value or not value.strip():
            raise HTTPException(status_code=400, detail=f"{field_name} must not be empty.")

    converted_time = pd.to_datetime(data.time_stamp, unit="ms", errors="coerce")
    if pd.isna(converted_time):
        raise HTTPException(status_code=400, detail="time_stamp must be a valid millisecond timestamp.")


@app.get("/")
def read_root():
    return {
        "message": "Online transportation fare estimation API is running.",
        "model_version": MODEL_VERSION,
        "note": "This is a local learning simulation, not a real production pricing system.",
    }


@app.post("/predict")
def predict_price(request: FarePredictionRequest):
    validate_request(request)

    if hasattr(request, "model_dump"):
        request_data = request.model_dump()
    else:
        request_data = request.dict()

    input_df = pd.DataFrame([request_data])
    input_df = create_time_features(input_df)
    prediction_input = input_df[FEATURE_COLUMNS]
    estimated_price = float(model.predict(prediction_input)[0])

    return {
        "estimated_price": round(estimated_price, 2),
        "model_version": MODEL_VERSION,
        "note": "This is a learning simulation for fare estimation, not a real production pricing system.",
    }
"""

request_example = {
    "distance": 2.5,
    "cab_type": "Uber",
    "source": "Back Bay",
    "destination": "North End",
    "name": "UberX",
    "time_stamp": 1544952607890,
}

api_readme = """# Local Fare Estimation API

Folder ini berisi skeleton API lokal FastAPI. Ini simulasi pembelajaran, bukan pricing system produksi.

## Install Library

```bash
pip install fastapi uvicorn joblib pandas scikit-learn
```

## Jalankan Lokal

Jalankan dari root folder:

```bash
uvicorn api.app:app --reload
```

Lalu buka:

```text
http://127.0.0.1:8000/
```

## Contoh Request

Kirim request `POST` ke:

```text
http://127.0.0.1:8000/predict
```

Contoh body:

```json
{
  "distance": 2.5,
  "cab_type": "Uber",
  "source": "Back Bay",
  "destination": "North End",
  "name": "UberX",
  "time_stamp": 1544952607890
}
```

Contoh yang sama ada di `api/request_example.json`.

## Contoh Response

```json
{
  "estimated_price": 10.75,
  "model_version": "v1.0-baseline-random-forest",
  "note": "This is a learning simulation for fare estimation, not a real production pricing system."
}
```

Harga estimasi bisa beda tergantung versi model.

## Keterbatasan

- API ini hanya demo lokal.
- Tidak di-deploy ke cloud.
- Tidak pakai `surge_multiplier`.
- Tidak butuh `price` karena itu target.
- Data cuaca belum digabung.
- Tidak boleh dipakai untuk menentukan harga nyata.
"""

(api_dir / "app.py").write_text(app_code, encoding="utf-8")
(api_dir / "request_example.json").write_text(json.dumps(request_example, indent=2), encoding="utf-8")
(api_dir / "README.md").write_text(api_readme, encoding="utf-8")

api_summary = pd.DataFrame({
    "API File": ["api/app.py", "api/request_example.json", "api/README.md"],
    "Created": [
        (api_dir / "app.py").exists(),
        (api_dir / "request_example.json").exists(),
        (api_dir / "README.md").exists(),
    ],
    "Purpose": [
        "Local FastAPI skeleton",
        "Example request body",
        "Local run instructions",
    ],
})

display(api_summary)







,API File,Created,Purpose
0,api/app.py,True,Local FastAPI skeleton
1,api/request_example.json,True,Example request body
2,api/README.md,True,Local run instructions


# Langkah 4: Skenario Shadow Deployment


In [5]:
cd_checklist = pd.DataFrame([
    {"CD Component": "Format model", "Design Choice": ".joblib", "Purpose": "Menyimpan pipeline sklearn", "Status": "READY" if model_file_available else "MISSING", "Notes": "Disimpan di models/baseline_price_model.joblib"},
    {"CD Component": "Model registry", "Design Choice": "models/model_registry.json", "Purpose": "Track model version, metrics, approval, and warnings", "Status": "READY" if registry_path.exists() else "MISSING", "Notes": "Simple JSON registry for mini project"},
    {"CD Component": "API framework", "Design Choice": "FastAPI", "Purpose": "Expose local prediction endpoint", "Status": "PLANNED", "Notes": "Local demo only, not cloud deployment"},
    {"CD Component": "Prediction endpoint", "Design Choice": "/predict", "Purpose": "Return estimated fare from trip input", "Status": "READY", "Notes": "Does not require price or surge_multiplier"},
    {"CD Component": "Deployment strategy", "Design Choice": "Shadow Deployment", "Purpose": "Test predictions in background before promotion", "Status": "DESIGNED", "Notes": "Safer simulation for pricing-related model"},
    {"CD Component": "Rollback strategy", "Design Choice": "Previous approved model version", "Purpose": "Return to earlier model if monitoring is unstable", "Status": "DESIGNED", "Notes": "Current project has one registered baseline version"},
    {"CD Component": "Monitoring metrics", "Design Choice": "MAE, RMSE, R2, group-level MAE", "Purpose": "Check accuracy and stability after release simulation", "Status": "DESIGNED", "Notes": "Service-name error warning must be monitored"},
    {"CD Component": "Logging plan", "Design Choice": "Input, output, timestamp, model version", "Purpose": "Support future monitoring and debugging", "Status": "PLANNED", "Notes": "No logging service is created in this step"},
    {"CD Component": "Safety note", "Design Choice": "Learning simulation only", "Purpose": "Avoid claiming real production pricing use", "Status": "DOCUMENTED", "Notes": "Not a real pricing system"},
])

display(cd_checklist)







,CD Component,Design Choice,Purpose,Status,Notes
0,Format model,.joblib,Menyimpan pipeline sklearn,READY,Disimpan di models/baseline_price_model.joblib
1,Model registry,models/model_registry.json,"Track model version, metrics, approval, and wa...",READY,Simple JSON registry for mini project
2,API framework,FastAPI,Expose local prediction endpoint,PLANNED,"Local demo only, not cloud deployment"
3,Prediction endpoint,/predict,Return estimated fare from trip input,READY,Does not require price or surge_multiplier
4,Deployment strategy,Shadow Deployment,Test predictions in background before promotion,DESIGNED,Safer simulation for pricing-related model
5,Rollback strategy,Previous approved model version,Return to earlier model if monitoring is unstable,DESIGNED,Current project has one registered baseline ve...
6,Monitoring metrics,"MAE, RMSE, R2, group-level MAE",Check accuracy and stability after release sim...,DESIGNED,Service-name error warning must be monitored
7,Logging plan,"Input, output, timestamp, model version",Support future monitoring and debugging,PLANNED,No logging service is created in this step
8,Safety note,Learning simulation only,Avoid claiming real production pricing use,DOCUMENTED,Not a real pricing system


# Langkah 5: Diagram Pipeline MLOps


In [6]:
def draw_pipeline_png(output_path):
    stages = [
        ("Data Layer", ["Raw Data", "Data Audit"]),
        ("CI Stage", ["CI Data\nValidation", "Data\nCleaning", "Feature\nEngineering", "Preprocessing\nPipeline"]),
        ("CT Stage", ["Model\nTraining", "Model\nEvaluation", "CT Quality\nChecklist"]),
        ("CD Stage", ["Model\nRegistry", "API\nService", "Shadow\nDeployment"]),
        ("Monitoring Stage", ["Monitoring", "Rollback or\nPromotion"]),
    ]

    fig, ax = plt.subplots(figsize=(18, 6))
    ax.set_xlim(0, 18)
    ax.set_ylim(0, 6)
    ax.axis("off")

    colors = ["#E8F4FA", "#EAF7EA", "#FFF4D8", "#F4EAFE", "#FDECEC"]
    box_color = "#FFFFFF"
    edge_color = "#2F3A45"

    x = 0.4
    y = 2.35
    box_w = 1.35
    box_h = 0.8
    gap = 0.28
    previous_center = None

    for stage_index, (stage_name, nodes) in enumerate(stages):
        stage_w = len(nodes) * box_w + (len(nodes) - 1) * gap + 0.35
        stage_x = x - 0.18
        ax.add_patch(Rectangle((stage_x, 1.45), stage_w, 3.2, facecolor=colors[stage_index], edgecolor="#B6C2CC", linewidth=1))
        ax.text(stage_x + stage_w / 2, 4.35, stage_name, ha="center", va="center", fontsize=11, fontweight="bold")

        for node in nodes:
            box = FancyBboxPatch(
                (x, y),
                box_w,
                box_h,
                boxstyle="round,pad=0.04,rounding_size=0.08",
                facecolor=box_color,
                edgecolor=edge_color,
                linewidth=1.2,
            )
            ax.add_patch(box)
            ax.text(x + box_w / 2, y + box_h / 2, node, ha="center", va="center", fontsize=9)

            current_center = (x + box_w / 2, y + box_h / 2)
            if previous_center is not None:
                ax.annotate(
                    "",
                    xy=(x, current_center[1]),
                    xytext=(previous_center[0] + box_w / 2, previous_center[1]),
                    arrowprops=dict(arrowstyle="->", color=edge_color, linewidth=1.2),
                )
            previous_center = current_center
            x += box_w + gap
        x += 0.35

    ax.text(9, 0.7, "CI validates data and preprocessing | CT validates model quality | CD prepares shadow deployment simulation", ha="center", fontsize=10)
    fig.tight_layout()
    fig.savefig(output_path, dpi=200, bbox_inches="tight")
    plt.close(fig)

try:
    draw_pipeline_png(diagram_png_path)
    png_status = "created"
except Exception as exc:
    png_status = f"not created: {exc}"

print("PNG diagram status:", png_status)
print("PNG diagram path:", diagram_png_path)







PNG diagram status: created
PNG diagram path: D:\SEMESTER 6\DATA MINING\MLOps-Tarif-Transportasi-Online\diagrams\mlops_pipeline_diagram.png


In [7]:
print("Ringkasan Akhir Langkah 4")
print("Cek file model:", model_file_available)
print("Registry model dibuat:", registry_path.exists())
print("Skeleton API dibuat:", (api_dir / "app.py").exists())
print("Laporan skenario CD dibuat:", cd_report_path.exists())
print("Diagram pipeline dibuat:", diagram_md_path.exists())
print("PNG pipeline dibuat:", diagram_png_path.exists())
print("Rekomendasi: siapkan laporan akhir dan presentasi.")







Ringkasan Akhir Langkah 4
Cek file model: True
Registry model dibuat: True
Skeleton API dibuat: True
Laporan skenario CD dibuat: False
Diagram pipeline dibuat: True
PNG pipeline dibuat: True
Rekomendasi: siapkan laporan akhir dan presentasi.
